# Build cell_features (Week 3)

Hand-crafted early-cycle features from `cycle_summary.csv`, merged with `voltage_features.csv`.

Labels (`EOL`, `initial_capacity`) come from `cell_targets.csv`.

Output: `data/processed/cell_features.csv` (134 rows, one per kept cell).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not find data/raw/ starting from {here}')


ROOT = find_repo_root()
TARGETS_PATH = ROOT / 'data' / 'cell_targets.csv'
SUMMARY_PATH = ROOT / 'data' / 'processed' / 'cycle_summary.csv'
VOLTAGE_PATH = ROOT / 'data' / 'processed' / 'voltage_features.csv'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'cell_features.csv'

WINDOWS = (20, 50, 100)
CAPACITY_CYCLES = (10, 50, 100)

print('Project root:', ROOT)
print('Output:', OUTPUT_PATH)

In [ ]:
def value_at_cycle(group: pd.DataFrame, cycle: int, column: str) -> float:
    row = group.loc[group['cycle_index'] == cycle, column]
    return float(row.iloc[0]) if len(row) else np.nan


def slope(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 2 or np.allclose(x, x[0]):
        return np.nan
    return float(np.polyfit(x, y, 1)[0])


def extract_summary_features(group: pd.DataFrame, initial_capacity: float) -> dict:
    g = group[group['cycle_index'] >= 1].sort_values('cycle_index')
    features = {}

    features['resistance_initial'] = value_at_cycle(g, 1, 'dc_internal_resistance')

    for cycle in CAPACITY_CYCLES:
        cap = value_at_cycle(g, cycle, 'discharge_capacity')
        features[f'capacity_c{cycle}'] = cap
        features[f'soh_c{cycle}'] = cap / initial_capacity if np.isfinite(cap) else np.nan

    for window in WINDOWS:
        w = g[g['cycle_index'] <= window]
        prefix = f'w{window}'
        if w.empty:
            for key in (
                f'capacity_slope_{prefix}',
                f'capacity_mean_{prefix}',
                f'capacity_std_{prefix}',
                f'soh_{prefix}',
                f'resistance_mean_{prefix}',
                f'resistance_slope_{prefix}',
                f'efficiency_mean_{prefix}',
                f'efficiency_std_{prefix}',
                f'temp_mean_{prefix}',
            ):
                features[key] = np.nan
            continue

        cyc = w['cycle_index'].to_numpy(dtype=float)
        cap = w['discharge_capacity'].to_numpy(dtype=float)
        features[f'capacity_slope_{prefix}'] = slope(cyc, cap)
        features[f'capacity_mean_{prefix}'] = float(np.nanmean(cap))
        features[f'capacity_std_{prefix}'] = float(np.nanstd(cap))

        cap_at_w = value_at_cycle(w, min(window, int(w['cycle_index'].max())), 'discharge_capacity')
        features[f'soh_{prefix}'] = cap_at_w / initial_capacity if np.isfinite(cap_at_w) else np.nan

        resistance = w['dc_internal_resistance'].to_numpy(dtype=float)
        ok_r = np.isfinite(resistance)
        features[f'resistance_mean_{prefix}'] = float(np.nanmean(resistance)) if ok_r.any() else np.nan
        features[f'resistance_slope_{prefix}'] = (
            slope(cyc[ok_r], resistance[ok_r]) if ok_r.sum() >= 2 else np.nan
        )

        efficiency = w['energy_efficiency'].to_numpy(dtype=float)
        ok_e = np.isfinite(efficiency)
        features[f'efficiency_mean_{prefix}'] = float(np.nanmean(efficiency)) if ok_e.any() else np.nan
        features[f'efficiency_std_{prefix}'] = float(np.nanstd(efficiency)) if ok_e.sum() >= 2 else np.nan

        temp = w['temperature_average'].to_numpy(dtype=float)
        ok_t = np.isfinite(temp)
        features[f'temp_mean_{prefix}'] = float(np.nanmean(temp)) if ok_t.any() else np.nan

    return features

In [ ]:
targets = pd.read_csv(TARGETS_PATH)
summary = pd.read_csv(SUMMARY_PATH)
voltage = pd.read_csv(VOLTAGE_PATH)

initial_caps = targets.set_index('file_id')['initial_capacity']
summary_parts = []
for file_id, group in summary.groupby('file_id', sort=True):
    summary_parts.append(
        {
            'file_id': file_id,
            'cell_id': group['cell_id'].iloc[0],
            **extract_summary_features(group, initial_caps[file_id]),
        }
    )

summary_features = pd.DataFrame(summary_parts)
cell_features = (
    targets.merge(summary_features, on=['file_id', 'cell_id'], how='inner')
    .merge(voltage, on=['file_id', 'cell_id'], how='inner')
)

label_cols = ['file_id', 'cell_id', 'EOL', 'initial_capacity']
feature_cols = [c for c in cell_features.columns if c not in label_cols]
cell_features = cell_features[label_cols + feature_cols]

print(f"Rows: {len(cell_features)}")
print(f"Summary features: {len(summary_features.columns) - 2}")
print(f"Voltage features: {len(voltage.columns) - 2}")
print(f"Total feature columns: {len(feature_cols)}")

In [ ]:
assert len(cell_features) == len(targets) == len(voltage)
assert cell_features['file_id'].is_unique
assert cell_features[feature_cols].isna().sum().sum() == 0

print('Missing values (should be 0 for current dataset):')
print(cell_features[feature_cols].isna().sum().sort_values(ascending=False).head())
cell_features.head()

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cell_features.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {OUTPUT_PATH}: {len(cell_features)} rows, {len(feature_cols)} feature columns")